# AutoSort Training Pipeline

Main steps:
1. Threshold detection
2. Training data preparation
3. Model training


In [1]:
import numpy as np
import pandas as pd
import pickle
import warnings
warnings.filterwarnings('ignore')
import os
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
from spikeinterface.core import concatenate_recordings
from probeinterface import write_probeinterface, read_probeinterface

import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from utils_clean import (
    prepare_training_data,
    train_autosort_model,
    CliqueInfo
)
import torch
import gc


In [ ]:
recording_path = "/media/ubuntu/sda/mouse_test/raw_data/WLF_128chmouse1_natima_RHD_251129_183351"
spike_inf_path = "/media/ubuntu/sda/mouse_test/sorted/WLF_128chmouse1_natima_RHD_251129_183351_10k//spike_inf.tsv"
neuron_inf_path = "/media/ubuntu/sda/mouse_test/sorted/WLF_128chmouse1_natima_RHD_251129_183351_10k//neuron_inf.pkl"

file_list = os.listdir(recording_path)
file_list.remove("settings.xml")
file_list = sorted(file_list)
recording_raw_list = []
for file in file_list:
    recording_raw_list.append(se.read_intan(f"{recording_path}/{file}", stream_id= '0'))
recording = concatenate_recordings(recording_list=recording_raw_list)
recording_raw = spre.unsigned_to_signed(recording)
recording_recorded = spre.bandpass_filter(recording_raw, freq_min=300, freq_max=3000)
recording_recorded = spre.notch_filter(recording_recorded, freq=50)
recording_f = spre.common_reference(recording_recorded, reference="global", operator="median")

# Load spike_inf and neuron_inf
spike_inf = pd.read_csv(spike_inf_path, sep='\t')
with open(neuron_inf_path, 'rb') as f:
    neuron_inf = pickle.load(f)

In [ ]:
probe = read_probeinterface('/media/ubuntu/sda/mouse_test/probe/tip_probe_128_1.json')
if probe is None:
    raise ValueError("Recording does not have probe information")

# 定义按shank构建cliques的函数
def build_shank_cliques(probe, shank_boundaries=[250, 750, 1250]):
    """
    根据x坐标划分shank并构建cliques
    
    Parameters:
        probe: Probe对象
        shank_boundaries: shank之间的x坐标边界，默认[250, 750, 1250]
                         将probe划分为4个shank:
                         - shank 0: x < 250
                         - shank 1: 250 <= x < 750
                         - shank 2: 750 <= x < 1250
                         - shank 3: x >= 1250
    
    Returns:
        cliques: List[CliqueInfo] - 每个shank对应一个clique
    """
    from typing import List
    
    df = probe.to_dataframe()
    if "device_channel_indices" in df.columns:
        device_indices = df["device_channel_indices"].astype(int).to_numpy()
    else:
        device_indices = np.arange(len(df), dtype=int)
    positions = df.loc[:, ["x", "y"]].to_numpy()
    contact_ids = df["contact_ids"].astype(str).to_numpy()
    
    # 根据x坐标划分shank
    x_coords = positions[:, 0]
    shank_boundaries_sorted = sorted(shank_boundaries)
    
    cliques: List[CliqueInfo] = []
    
    # 定义shank范围
    shank_ranges = [
        (float('-inf'), shank_boundaries_sorted[0]),  # shank 0: x < 250
        (shank_boundaries_sorted[0], shank_boundaries_sorted[1]),  # shank 1: 250 <= x < 750
        (shank_boundaries_sorted[1], shank_boundaries_sorted[2]),  # shank 2: 750 <= x < 1250
        (shank_boundaries_sorted[2], float('inf')),  # shank 3: x >= 1250
    ]
    
    for shank_id, (x_min, x_max) in enumerate(shank_ranges):
        # 找到属于当前shank的通道
        if x_min == float('-inf'):
            mask = x_coords < x_max
        elif x_max == float('inf'):
            mask = x_coords >= x_min
        else:
            mask = (x_coords >= x_min) & (x_coords < x_max)
        
        shank_device_indices = device_indices[mask]
        shank_contact_ids = contact_ids[mask]
        shank_positions = positions[mask]
        
        if len(shank_device_indices) == 0:
            print(f"[WARNING] Shank {shank_id} has no channels")
            continue
        
        # 计算shank的中心位置
        center = tuple(np.mean(shank_positions, axis=0))
        
        # 创建CliqueInfo对象
        clique = CliqueInfo(
            clique_id=shank_id,
            device_channel_indices=list(shank_device_indices),
            contact_ids=list(shank_contact_ids),
            center=center,
        )
        cliques.append(clique)
        
        print(f"[INFO] Shank {shank_id}: {len(shank_device_indices)} channels "
              f"(x range: {x_min if x_min != float('-inf') else 'min'} to "
              f"{x_max if x_max != float('inf') else 'max'})")
    
    print(f"[INFO] Built {len(cliques)} cliques from {len(shank_boundaries) + 1} shanks")
    return cliques

# Build cliques from probe by shank
cliques = build_shank_cliques(probe, shank_boundaries=[250, 750, 1250])

# Plot cliques visualization
output_dir = Path("/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/clique_128_kilosort//output")
output_dir.mkdir(parents=True, exist_ok=True)
#plot_cliques(probe, cliques, output_pdf_path=str(output_dir / "cliques_visualization.pdf"))

# Save clique information for evaluation
clique_info = {
    'cliques': cliques,  # List of CliqueInfo objects
    'clique_params': {
        'method': 'shank_based',
        'shank_boundaries': [250, 750, 1250],
        'num_shanks': 4,
    },
    'probe_df': probe.to_dataframe(),  # Probe dataframe for verification
    'recording_path': recording_path,  # Recording path for reference
}

clique_info_path = output_dir / "clique_info.pkl"
with open(clique_info_path, 'wb') as f:
    pickle.dump(clique_info, f)

[INFO] Shank 0: 32 channels (x range: min to 250)
[INFO] Shank 1: 32 channels (x range: 250 to 750)
[INFO] Shank 2: 32 channels (x range: 750 to 1250)
[INFO] Shank 3: 32 channels (x range: 1250 to max)
[INFO] Built 4 cliques from 4 shanks


In [ ]:
# Set parameters
base_save_dir = "/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/clique_128_kilosort/"
duration_seconds = 200  # Processing duration (seconds)

# Detection parameters (consistent with AutoSort default values)
detection_params = {
    'thr_min': 3.5,
    'thr_max': 30,
    'distance': 3,
    'ch_max_simul_firing': 6,
    'wlen': 15,
    'prominence': 10,
}

# Waveform window parameters
window_params = {
    'left_sample': 10,
    'right_sample': 20,
}

# Get recording channel IDs (to map device_channel_indices to channel IDs)
recording_channel_ids = recording_f.get_channel_ids()

# Process each clique independently
train_data_dirs = {}  # Store train_data_dir for each clique

for clique in cliques:
    clique_id = clique.clique_id
    clique_channel_indices = clique.device_channel_indices
    
    print(f"\n{'='*80}")
    print(f"Processing Clique {clique_id}")
    print(f"{'='*80}")
    print(f"Clique {clique_id} has {len(clique_channel_indices)} channels")
    print(f"Channel indices: {clique_channel_indices}")
    
    # Get channel IDs for this clique (device_channel_indices correspond to recording channel indices)
    clique_channel_ids = [recording_channel_ids[idx] for idx in clique_channel_indices]
    print(f"Channel IDs: {clique_channel_ids}")
    
    # Slice recording to only include channels in this clique
    recording_clique = recording_f.select_channels(channel_ids=clique_channel_ids)
    print(f"Sliced recording has {recording_clique.get_num_channels()} channels")
    
    # Filter neuron_inf: neuron belongs to clique if ALL its channel_ids are in clique's channel_indices
    # neuron_inf has 'Neuron' column and 'channel_id' column (list of channel indices)
    import ast
    import re
    
    def neuron_belongs_to_clique(row, clique_channel_indices_set):
        """Check if all channel_ids of a neuron are in clique's channel_indices"""
        channel_id = row['channel_id']
        
        # Parse channel_id if it's a string representation of list
        if isinstance(channel_id, str):
            try:
                channel_id = ast.literal_eval(channel_id)
            except:
                # If parsing fails, try to extract numbers
                channel_id = [int(x) for x in re.findall(r'\d+', channel_id)]
        
        # Convert to list if it's not already
        if not isinstance(channel_id, (list, tuple, np.ndarray)):
            channel_id = [channel_id]
        else:
            channel_id = list(channel_id)
        
        # Check if all channel_ids are in clique
        return all(ch in clique_channel_indices_set for ch in channel_id)
    
    clique_channel_indices_set = set(clique_channel_indices)
    neuron_mask = neuron_inf.apply(lambda row: neuron_belongs_to_clique(row, clique_channel_indices_set), axis=1)
    neuron_inf_clique = neuron_inf[neuron_mask].copy()
    print(f"Filtered neuron_inf: {len(neuron_inf_clique)} neurons in clique {clique_id}")
    
    if len(neuron_inf_clique) == 0:
        print(f"[WARNING] Clique {clique_id} has no neurons, skipping...")
        continue
    
    # Filter spike_inf to only include spikes from neurons in this clique
    # neuron_inf uses 'Neuron', spike_inf uses 'neuron'
    valid_neurons = neuron_inf_clique['Neuron'].unique()
    spike_inf_clique = spike_inf[spike_inf['neuron'].isin(valid_neurons)].copy()
    print(f"Filtered spike_inf: {len(spike_inf_clique)} spikes in clique {clique_id}")
    
    # Extract valid channels for this clique (channels that have neurons)
    # Collect all channel_ids from neurons in this clique
    all_channel_ids = []
    for _, row in neuron_inf_clique.iterrows():
        channel_id = row['channel_id']
        if isinstance(channel_id, str):
            try:
                channel_id = ast.literal_eval(channel_id)
            except:
                import re
                channel_id = [int(x) for x in re.findall(r'\d+', channel_id)]
        if not isinstance(channel_id, (list, tuple, np.ndarray)):
            channel_id = [channel_id]
        all_channel_ids.extend(list(channel_id))
    
    valid_channels_clique = sorted(set(all_channel_ids))
    # Map to clique-local channel indices (0, 1, 2, ...)
    # Since we sliced the recording, the channel indices in the sliced recording are 0, 1, 2, ...
    # We need to map tract_channel (which is the original device_channel_index) to the new local index
    channel_index_mapping = {orig_idx: local_idx for local_idx, orig_idx in enumerate(clique_channel_indices)}
    valid_channels_clique_local = [channel_index_mapping[ch] for ch in valid_channels_clique if ch in channel_index_mapping]
    
    print(f"Valid channels (original indices): {valid_channels_clique}")
    print(f"Valid channels (local indices in sliced recording): {valid_channels_clique_local}")
    
    # IMPORTANT: Map tract_channel in neuron_inf_clique to local channel indices
    # This ensures that gt_array and detect_array use the same channel indexing scheme
    # gt_array uses tract_channel, detect_array uses local channel indices (0, 1, 2, ...)
    neuron_inf_clique_mapped = neuron_inf_clique.copy()
    if 'tract_channel' in neuron_inf_clique_mapped.columns:
        # Map tract_channel from original indices to local indices
        def map_tract_channel(orig_ch):
            return channel_index_mapping.get(orig_ch, orig_ch)  # If not in mapping, keep original
        
        neuron_inf_clique_mapped['tract_channel'] = neuron_inf_clique_mapped['tract_channel'].apply(map_tract_channel)
        print(f"Mapped tract_channel to local indices for gt_array matching")
    else:
        # If tract_channel doesn't exist, try to use channel_id's first channel
        print(f"[WARNING] neuron_inf_clique doesn't have 'tract_channel' column")
        # Try to create tract_channel from channel_id
        if 'channel_id' in neuron_inf_clique_mapped.columns:
            def get_first_channel(channel_id):
                if isinstance(channel_id, str):
                    try:
                        channel_id = ast.literal_eval(channel_id)
                    except:
                        import re
                        channel_id = [int(x) for x in re.findall(r'\d+', channel_id)]
                if not isinstance(channel_id, (list, tuple, np.ndarray)):
                    channel_id = [channel_id]
                if len(channel_id) > 0:
                    orig_ch = channel_id[0]
                    return channel_index_mapping.get(orig_ch, orig_ch)
                return None
            
            neuron_inf_clique_mapped['tract_channel'] = neuron_inf_clique_mapped['channel_id'].apply(get_first_channel)
            print(f"Created tract_channel from channel_id's first channel")
    
    # Create save directory for this clique
    save_dir_clique = base_save_dir + f"clique_{clique_id:02d}/"
    
    # Prepare training data for this clique
    train_data_dir_clique = prepare_training_data(
        recording_f=recording_clique,
        spike_inf=spike_inf_clique,
        neuron_inf=neuron_inf_clique_mapped,  # Use mapped neuron_inf
        save_dir=save_dir_clique,
        duration_seconds=duration_seconds,
        valid_channels=valid_channels_clique_local,  # Use local channel indices
        **detection_params,
        **window_params
    )
    
    train_data_dirs[clique_id] = train_data_dir_clique
    print(f"Clique {clique_id} training data saved to: {train_data_dir_clique}")

print(f"\n{'='*80}")
print(f"All cliques processed! Total: {len(train_data_dirs)} cliques")
print(f"{'='*80}")



Processing Clique 0
Clique 0 has 32 channels
Channel indices: [np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(14), np.int64(15), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(36), np.int64(47), np.int64(48), np.int64(50), np.int64(51), np.int64(52), np.int64(53), np.int64(55), np.int64(56), np.int64(57), np.int64(58), np.int64(59), np.int64(61)]
Channel IDs: [np.str_('B-008'), np.str_('B-009'), np.str_('B-010'), np.str_('B-011'), np.str_('B-014'), np.str_('B-015'), np.str_('B-020'), np.str_('B-021'), np.str_('B-022'), np.str_('B-023'), np.str_('B-024'), np.str_('B-025'), np.str_('B-026'), np.str_('B-027'), np.str_('B-030'), np.str_('B-031'), np.str_('B-032'), np.str_('B-033'), np.str_('B-034'), np.str_('B-036'), np.str_('B-047'), np.str_('B-048'), np.str_('B-050'), np.str_('B-051'), np.str_('B-052'), np.str_('B-053'), np.str_(

Extracting waveforms: 100%|██████████| 30/30 [00:10<00:00,  2.88it/s]


Waveform extraction completed!
waveform shape: (78950, 32, 30)

### 4. Save Training Data
Save directory: /media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/clique_128_kilosort/clique_00/train_data
  ✓ neuron_mapping.pkl saved
Saving data...
  ✓ X_waveform.pkl saved
  ✓ Y_spike_id.pkl saved
  ✓ Y_spike_id_noise.pkl saved
  ✓ X_spiketrain_time.pkl saved

All data saved to: /media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/clique_128_kilosort/clique_00/train_data
Data statistics:
  - Total spike count: 78950
  - Number of channels: 32
  - Window length: 30
  - Number of unique units: 15
  - Noise spike count: 74508
  - Valid spike count: 4442
Clique 0 training data saved to: /media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/clique_128_kilosort/clique_00/train_data

Processing Clique 1
Clique 1 has 32 channels
Channel indices: [np.int64(0), np.int64(1), np.int

Extracting waveforms: 100%|██████████| 30/30 [00:17<00:00,  1.67it/s]


Waveform extraction completed!
waveform shape: (148108, 32, 30)

### 4. Save Training Data
Save directory: /media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/clique_128_kilosort/clique_01/train_data
  ✓ neuron_mapping.pkl saved
Saving data...
  ✓ X_waveform.pkl saved
  ✓ Y_spike_id.pkl saved
  ✓ Y_spike_id_noise.pkl saved
  ✓ X_spiketrain_time.pkl saved

All data saved to: /media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/clique_128_kilosort/clique_01/train_data
Data statistics:
  - Total spike count: 148108
  - Number of channels: 32
  - Window length: 30
  - Number of unique units: 23
  - Noise spike count: 111412
  - Valid spike count: 36696
Clique 1 training data saved to: /media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/clique_128_kilosort/clique_01/train_data

Processing Clique 2
Clique 2 has 32 channels
Channel indices: [np.int64(64), np.int64(65), 

Extracting waveforms: 100%|██████████| 30/30 [00:01<00:00, 27.82it/s]


Waveform extraction completed!
waveform shape: (37349, 32, 30)

### 4. Save Training Data
Save directory: /media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/clique_128_kilosort/clique_02/train_data
  ✓ neuron_mapping.pkl saved
Saving data...
  ✓ X_waveform.pkl saved
  ✓ Y_spike_id.pkl saved
  ✓ Y_spike_id_noise.pkl saved
  ✓ X_spiketrain_time.pkl saved

All data saved to: /media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/clique_128_kilosort/clique_02/train_data
Data statistics:
  - Total spike count: 37349
  - Number of channels: 32
  - Window length: 30
  - Number of unique units: 2
  - Noise spike count: 33502
  - Valid spike count: 3847
Clique 2 training data saved to: /media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/clique_128_kilosort/clique_02/train_data

Processing Clique 3
Clique 3 has 32 channels
Channel indices: [np.int64(66), np.int64(68), np.in

Extracting waveforms: 100%|██████████| 30/30 [00:08<00:00,  3.42it/s]


Waveform extraction completed!
waveform shape: (350673, 32, 30)

### 4. Save Training Data
Save directory: /media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/clique_128_kilosort/clique_03/train_data
  ✓ neuron_mapping.pkl saved
Saving data...
  ✓ X_waveform.pkl saved
  ✓ Y_spike_id.pkl saved
  ✓ Y_spike_id_noise.pkl saved
  ✓ X_spiketrain_time.pkl saved

All data saved to: /media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/clique_128_kilosort/clique_03/train_data
Data statistics:
  - Total spike count: 350673
  - Number of channels: 32
  - Window length: 30
  - Number of unique units: 29
  - Noise spike count: 301976
  - Valid spike count: 48697
Clique 3 training data saved to: /media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/clique_128_kilosort/clique_03/train_data

All cliques processed! Total: 4 cliques


## Step 2: Model Training


In [5]:
# Set training parameters
training_params = {
    'epochs': 20,
    'batch_size': 512,
    'left_sample': 10,
    'right_sample': 20,
    'early_stopping': True,  # Enable early stopping
    'patience': 5,  # Stop if accuracy doesn't improve for 5 consecutive epochs
    'min_delta': 0.0,  # Minimum change
}

# Repeat training 5 times for each clique
n_runs = 5

# Train models for each clique
all_models_dict = {}  # {clique_id: [models]}
all_logs_dict = {}    # {clique_id: [logs]}

for clique_id, train_data_dir_clique in train_data_dirs.items():
    print(f"\n{'='*80}")
    print(f"Training models for Clique {clique_id}")
    print(f"{'='*80}")
    
    # Get the number of channels for this clique's recording
    # We need to reload the sliced recording or get it from the clique info
    clique = cliques[clique_id]
    n_channels_clique = len(clique.device_channel_indices)
    
    base_model_save_dir_clique = base_save_dir + f"clique_{clique_id:02d}/model_save/"
    
    all_models_clique = []
    all_logs_clique = []
    
    for run_id in range(1, n_runs + 1):
        print(f"\n{'='*60}")
        print(f"Clique {clique_id} - Training run {run_id}/{n_runs}")
        print(f"{'='*60}")
        
        # Create independent save directory for each training run
        model_save_dir_clique = base_model_save_dir_clique + f"run_{run_id}/"
        
        # Train model
        autosort_model, training_log = train_autosort_model(
            train_data_dir=train_data_dir_clique,
            model_save_dir=model_save_dir_clique,
            n_channels=n_channels_clique,
            **training_params
        )
        
        all_models_clique.append(autosort_model)
        all_logs_clique.append(training_log)
        
        print(f"\nClique {clique_id} - Training run {run_id} completed!")
        print(f"Model save directory: {model_save_dir_clique}")
    
    all_models_dict[clique_id] = all_models_clique
    all_logs_dict[clique_id] = all_logs_clique
    
    print(f"\n{'='*60}")
    print(f"Clique {clique_id} - All {n_runs} training runs completed!")
    print(f"{'='*60}")

print(f"\n{'='*80}")
print(f"All training completed for all cliques!")
print(f"Total cliques trained: {len(all_models_dict)}")
print(f"{'='*80}")



Training models for Clique 0

Clique 0 - Training run 1/5
Using device: cuda
Create dataset...
Dataset loaded:
  - Total samples: 78950
  - Number of channels: 32
  - Window length: 30
  - Number of unique units: 15
  - Noise samples: 74508.0
  - Non-noise samples: 4442.0
Model parameters:
  - Number of channels: 32
  - Window length: 30
  - Number of units: 15
  - Input dimension: 990
Unit ID list saved to: /media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/clique_128_kilosort/clique_00/model_save/run_1/keep_id.pkl

Dataset split:
  - Training set: 63160 samples
  - Validation set: 15790 samples

Starting training (total 20 epochs)...
Early stopping enabled: patience=5, min_delta=0.0
epoch : 1/20


Training: 100%|██████████| 124/124 [00:01<00:00, 100.66it/s]


epoch : 1/20, detection loss = 542.900471, classification loss = 1052.221475


Validation: 100%|██████████| 31/31 [00:00<00:00, 222.58it/s]


epoch : 1/20, val detection loss = 462.262402, classification loss = 887.301658
Validation Loss Decreased(inf--->1349.564060)
Validation Accuracy Decreased(inf--->0.640659) 	 Saving The Model (Best Acc Epoch)
epoch : 2/20


...

epoch : 5/20, val detection loss = 264.517462, classification loss = 236.567729
Validation Loss Decreased(570.431244--->501.085191)
epoch : 6/20


Training: 100%|██████████| 548/548 [00:03<00:00, 146.47it/s]


epoch : 6/20, detection loss = 182.480464, classification loss = 124.330636


Validation: 100%|██████████| 137/137 [00:00<00:00, 234.60it/s]


epoch : 6/20, val detection loss = 273.162500, classification loss = 241.586728

Early stopping triggered: 5 consecutive epochs without improvement
Best accuracy: 0.867156 (Epoch 1)

Training completed! Training log saved to: /media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/clique_128_kilosort/clique_03/model_save/run_5/training_log.csv
Best validation accuracy: 0.867156 (Epoch 1)

Clique 3 - Training run 5 completed!
Model save directory: /media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/clique_128_kilosort/clique_03/model_save/run_5/

Clique 3 - All 5 training runs completed!

All training completed for all cliques!
Total cliques trained: 4
